# Noble AI — Laboratorio integral Colab v1.2

Ejecuta **Entorno de ejecución → Ejecutar todas**. Este notebook clonará Noble AI, instalará dependencias, compilará frontend, ejecutará tests y, cuando haya NVIDIA/CUDA, probará un GGUF auténtico.

**Diagnóstico visible:** cada etapa enseña hora de inicio, estado, últimas líneas de salida, tiempo transcurrido y avisos cada 12 segundos cuando un comando no imprime nada. Cada comando tiene un límite de ejecución. Se genera un ZIP con resultados incluso cuando fallan etapas individuales.

La app solo escucha en localhost de Colab. No se publican túneles ni puertos públicos. El notebook se puede ejecutar sin GPU (la inferencia quedará SKIP). Si detuviste el notebook anterior tras 30 minutos, puedes usar este nuevo cuaderno sin depender del archivo descargable anterior.

In [ ]:
RUN_GPU = True             # GPU real con modelo GGUF, si está disponible
RUN_BROWSER = True         # Playwright escritorio/móvil
RUNS = 3                   # 1 a 10 generaciones narrativas
KEEP_APP_RUNNING = True
MODEL_REPO = "bartowski/Qwen_Qwen3-4B-Instruct-2507-GGUF"
MODEL_FILE = "Qwen3-4B-Instruct-2507-Q4_K_M.gguf"
API_PORT = 8181            # Evita conflicto con el notebook anterior
LLM_PORT = 5181

from pathlib import Path
import os, sys, json, time, subprocess, shutil, zipfile, re, threading, traceback
from datetime import datetime, timezone

BASE = Path("/content/noble_qa_v12")
ROOT = BASE / "novel_ai"
EVIDENCE = BASE / "diagnostico"
LOGS = EVIDENCE / "logs"
ZIP_FILE = Path("/content/Noble_AI_Colab_QA_v1_2.zip")
LOGS.mkdir(parents=True, exist_ok=True)
T0 = time.monotonic()
state = {"version": "1.2", "stages": {}, "environment": {}, "benchmarks": {},
         "started_utc": datetime.now(timezone.utc).isoformat()}
children = {}

def clock():
    return datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
def clean(value):
    text = str(value)
    text = re.sub(r"(?i)(Bearer\s+)[^\s\"']+", r"\1[REDACTED]", text)
    text = re.sub(r"(?i)((?:api[_-]?key|password|secret|token)\s*[:=]\s*)[^\s\"']+", r"\1[REDACTED]", text)
    return text
def save_status():
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    state["elapsed_seconds"] = round(time.monotonic() - T0, 1)
    (EVIDENCE / "report.json").write_text(json.dumps(state, indent=2, ensure_ascii=False), encoding="utf-8")
    lines = [f"{name} — {result['status']}: {result.get('detail','')}"
             for name, result in state["stages"].items()]
    (EVIDENCE / "ESTADO_ACTUAL.txt").write_text(
        f"{clock()} | {state['elapsed_seconds']} segundos\n" + "\n".join(lines) + "\n",
        encoding="utf-8")
def update(name, status, detail=""):
    state["stages"][name] = {"status": status, "detail": clean(detail)[:360],
                             "updated_utc": clock()}
    print(f"[{clock()}] [{status}] {name}: {clean(detail)[:240]}", flush=True)
    save_status()
def skip(name, why):
    update(name, "SKIP", why)

def command(name, argv, cwd=None, timeout=600, environment=None, watch=None):
    """Logs en directo. Nunca capture_output para instalaciones largas."""
    update(name, "RUNNING", f"Ejecutando (límite {timeout}s): " + " ".join(argv[:3]))
    logfile = LOGS / (name + ".log")
    began = time.monotonic()
    last = began
    offset = 0
    fragment = ""
    proc = None
    try:
        with logfile.open("wb") as sink:
            proc = subprocess.Popen(argv, cwd=str(cwd or BASE), env=environment,
                                    stdout=sink, stderr=subprocess.STDOUT,
                                    stdin=subprocess.DEVNULL, start_new_session=True)
            while proc.poll() is None:
                time.sleep(2)
                now = time.monotonic()
                with logfile.open("rb") as reader:
                    reader.seek(offset)
                    chunk = reader.read(200000)
                    offset = reader.tell()
                if chunk:
                    fragment += chunk.decode("utf-8", errors="replace")
                    lines = fragment.splitlines(keepends=True)
                    fragment = "" if not lines or lines[-1].endswith(("\n","\r")) else lines.pop()
                    latest = [clean(line.strip()) for line in lines if line.strip()]
                    for line in latest[-4:]:
                        print("   [" + name + "] " + line[:240], flush=True)
                    if latest: last = now
                if now - last >= 12:
                    extra = ""
                    if watch and Path(watch).exists():
                        try:
                            amount = sum(f.stat().st_size for f in Path(watch).rglob("*") if f.is_file())
                            extra = f" | archivos ~{amount/1048576:.1f} MB"
                        except OSError: pass
                    print(f"   [ACTIVO] {name}: {int(now-began)}s de {timeout}s límite{extra}", flush=True)
                    state["stages"][name]["running_seconds"] = round(now-began, 1)
                    save_status()
                    last = now
                if now - began >= timeout:
                    try:
                        import signal
                        os.killpg(proc.pid, signal.SIGTERM)
                        proc.wait(timeout=5)
                    except (OSError, subprocess.TimeoutExpired):
                        proc.kill(); proc.wait()
                    update(name, "FAIL", f"Se superó el límite de {timeout}s. Log: {logfile.name}")
                    return False
        if fragment.strip(): print("   ["+name+"] "+clean(fragment.strip())[:240], flush=True)
        ok = proc.returncode == 0
        if not ok:
            for line in logfile.read_text(errors="replace").splitlines()[-8:]:
                print("   [ERROR] " + clean(line)[:230], flush=True)
        update(name, "PASS" if ok else "FAIL",
               f"Código {proc.returncode}, duración {int(time.monotonic()-began)}s, log {logfile.name}")
        return ok
    except Exception as error:
        update(name, "FAIL", type(error).__name__ + ": " + str(error)[:190])
        return False
    finally:
        if proc is not None and proc.poll() is None:
            proc.terminate()
        if logfile.exists():
            try:
                with logfile.open("rb") as src:
                    src.seek(max(0, logfile.stat().st_size - 450000))
                    data = src.read().decode(errors="replace")
                logfile.write_text(clean(data), encoding="utf-8")
            except OSError: pass

def gpu_snapshot():
    if not shutil.which("nvidia-smi"): return []
    try:
        proc = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.used,utilization.gpu",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=7, check=True)
        result = []
        for line in proc.stdout.splitlines():
            parts = [part.strip() for part in line.split(",")]
            if len(parts)==4:
                result.append({"name":parts[0],"total_mb":int(parts[1]),
                   "used_mb":int(parts[2]),"util_percent":int(parts[3])})
        return result
    except Exception: return []

def http_json(url, data=None, method="GET", timeout=10):
    import urllib.request
    raw = json.dumps(data).encode() if data is not None else None
    req = urllib.request.Request(url, data=raw, method=method,
          headers={"Content-Type":"application/json"} if raw is not None else {})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return json.load(response)

save_status()
print("Noble AI QA v1.2",clock(),"· Registro de cada etapa y aviso cada 12 segundos.",flush=True)

## 1. Descargar Noble AI y reconocer el hardware

El estado **PASS/FAIL** se muestra en esta celda. Los logs se guardan en `diagnostico/logs/`.

In [ ]:
if (ROOT / ".git").is_dir():
    REPO_OK = command("git_update", ["git","-C",str(ROOT),"pull","--ff-only"], timeout=240)
else:
    REPO_OK = command("git_clone", ["git","clone","--progress","--depth","1",
          "https://github.com/Mairon31/novel_ai.git",str(ROOT)], timeout=480)
if REPO_OK:
    version = subprocess.run(["git","-C",str(ROOT),"rev-parse","HEAD"],
                             capture_output=True,text=True,timeout=12)
    state["environment"]["commit"] = version.stdout.strip()
    update("git_revision","PASS",version.stdout.strip())
else:
    skip("git_revision","Sin repositorio disponible")
state["environment"]["python"] = sys.version.split()[0]
state["environment"]["gpu_start"] = gpu_snapshot()
state["environment"]["disk_free_gib"] = round(shutil.disk_usage(BASE).free/1024**3,2)
save_status()
print("Hardware NVIDIA detectado:", state["environment"]["gpu_start"], flush=True)

## 2. Instalar dependencias, compilar y ejecutar los tests Python

Las pruebas y la compilación se ejecutan con límites de tiempo independientes. Si una etapa falla, se registra el problema y continúa con lo que pueda verificarse.

In [ ]:
PY_READY = False
WEB_READY = False
if REPO_OK:
    PY_READY = command("python_deps",
        [sys.executable,"-m","pip","install","-r",str(ROOT/"apps/api/requirements.txt")],
        timeout=900)
    if PY_READY:
        command("python_tests",[sys.executable,"-u","-m","unittest","discover",
           "-s","tests","-p","test_*.py","-v"],cwd=ROOT,timeout=480)
    else:
        skip("python_tests","Falló la instalación Python")
    try:
        node = subprocess.run(["node","--version"],capture_output=True,text=True,timeout=8)
        major = int(node.stdout.strip().lstrip("v").split(".")[0])
        NODE_READY = major>=22
    except Exception:
        NODE_READY = False
    if not NODE_READY:
        if command("node_manager",["npm","install","-g","n","--no-audit","--no-fund"],timeout=360):
            NODE_READY = command("node_22",["n","22"],timeout=480)
    if NODE_READY:
        web = ROOT/"apps/web"
        dependencies = command("web_deps",["npm","ci","--no-audit","--no-fund"],cwd=web,timeout=900)
        if dependencies:
            WEB_READY = command("web_build",["npm","run","build"],cwd=web,timeout=600)
        else:
            skip("web_build","Fallaron las dependencias web")
    else:
        skip("web_build","Node.js 22 no disponible")
else:
    for stage in ("python_deps","python_tests","web_deps","web_build"):
        skip(stage,"Falló la obtención del repositorio")
save_status()

## 3. Desplegar Noble AI en el entorno Colab

El backend sirve la compilación de React por `127.0.0.1:8181`. Su arranque tiene un límite de espera y guarda logs del servidor, en lugar de bloquear la ejecución sin explicación.

In [ ]:
API_READY = False
if PY_READY:
    server_log = (LOGS/"noble_api.log").open("ab", buffering=0)
    environment = os.environ.copy()
    environment["NOBLE_DATA_DIR"] = str(BASE/"workspace_de_prueba")
    children["api"] = subprocess.Popen(
        [sys.executable,"-u","-m","uvicorn","noble_api.main:app",
         "--app-dir",str(ROOT/"apps/api"),"--host","127.0.0.1","--port",str(API_PORT)],
        cwd=str(ROOT),env=environment,stdout=server_log,
        stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,start_new_session=True)
    update("api_deploy","RUNNING","Esperando /health hasta 90 segundos")
    for i in range(90):
        try:
            health = http_json(f"http://127.0.0.1:{API_PORT}/api/v1/health",timeout=3)
            if health.get("status") == "ok":
                API_READY = True
                break
        except Exception: pass
        if children["api"].poll() is not None:
            break
        if i%12==11:
            print(f"   [ACTIVO] FastAPI {i+1}s | proceso {children['api'].poll()} | log noble_api.log",flush=True)
        time.sleep(1)
    update("api_deploy","PASS" if API_READY else "FAIL",
           "Servidor HTTP local listo" if API_READY else "No respondió: revisar noble_api.log")
else:
    skip("api_deploy","Dependencias Python no disponibles")
if API_READY and WEB_READY:
    try:
        import urllib.request
        with urllib.request.urlopen(f"http://127.0.0.1:{API_PORT}/",timeout=10) as r:
            html = r.read(5000).decode(errors="replace").lower()
        update("ui_deploy","PASS" if "<html" in html else "FAIL",
               "React compilado servido por FastAPI")
    except Exception as exc:
        update("ui_deploy","FAIL",type(exc).__name__+": "+str(exc)[:150])
else:
    skip("ui_deploy","API o frontend no disponible")

## 4. QA visual de escritorio y móvil

Instala Chromium y ejecuta las pruebas de Playwright contra la versión compilada. El resultado incluye archivos de `test-results` cuando estén disponibles.

In [ ]:
if RUN_BROWSER and API_READY and WEB_READY:
    web = ROOT/"apps/web"
    chromium = command("playwright_install",
        ["npx","playwright","install","chromium","--with-deps"],
        cwd=web,timeout=1200)
    if chromium:
        environment = os.environ.copy()
        environment["NOBLE_E2E_BASE_URL"] = f"http://127.0.0.1:{API_PORT}"
        command("browser_tests",["npm","run","test:e2e"],
                cwd=web,timeout=900,environment=environment)
        source = web/"test-results"
        if source.is_dir():
            shutil.copytree(source,EVIDENCE/"browser-artifacts",dirs_exist_ok=True)
    else:
        skip("browser_tests","No se instaló Playwright Chromium")
else:
    skip("browser_tests","Desactivado o despliegue incompleto")
save_status()